# Baseline CNNs on Tiny-ImageNet — FP32 & QAT (INT8)

**Author:** Rafael  
**Dataset:** Tiny-ImageNet-200 (200 classes, 64×64 RGB)  
**Goal:** Train and compare the 4 Phase 1 reference architectures in FP32, then apply
Quantization-Aware Training (QAT) to the QAT-compatible models to obtain INT8 versions.

| Model | QAT | Notes |
|---|---|---|
| AlexNetTV | ✓ | Pretrained torchvision AlexNet, Conv-ReLU fused |
| VGGStyleCNN | ✓ | From-scratch, stacked 3×3, Conv-BN-ReLU stages |
| ResNet18TV | ✗ | Pretrained ResNet-18; residual adds lack FloatFunctional → FP32 only |
| MobileNetV2TV | ✗ | Pretrained MobileNetV2; inverted residual adds → FP32 only |

## Notebook layout

1. Imports & reproducibility
2. Dataset & loaders
3. Model shape check
4. Model registration (fuse maps)
5. FP32 training
6. QAT training (AlexNetTV, VGGStyleCNN)
7. INT8 conversion & CPU evaluation
8. FP32 evaluation (reload best checkpoints)
9. Final comparison table
10. Persist experiment summary

## 1. Imports & reproducibility


In [1]:
import os
import random
import sys
from dataclasses import asdict, replace
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torchinfo
import wandb

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from ml import (
    DataConfig, TrainerConfig, QATConfig,
    create_imagenet_loaders,
    MODEL_REGISTRY, register_model,
    Trainer, make_qat_callback,
    build_qat, load_best_model, convert_to_int8,
    build_comparison_table, create_results_summary, disk_mb,
)
from configs.loader import load_config

from models import (
    build_alexnet, AlexNetTV,
    build_vggstylecnn, VGGStyleCNN,
    build_resnet18, ResNet18TV,
    build_mobilenetv2, MobileNetV2TV,
)

torch.backends.quantized.engine = "fbgemm"

/home/rafael/Documents/alexnet_rafael/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SAVE_DIR    = project_root / "checkpoints" / "baselines_qat_phase1"
RESULTS_DIR = project_root / "results" / "baselines_qat_phase1"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

data_cfg = DataConfig(**load_config("data.yaml"))
data_cfg.seed = SEED
fp32_cfg = TrainerConfig(**load_config("training.yaml"))
qat_cfg  = QATConfig(**load_config("qat.yaml"))
print(device)

cuda


## 2. Dataset & loaders


In [3]:
import kagglehub

dataset_path = kagglehub.dataset_download("akash2sharma/tiny-imagenet")
train_path = os.path.join(dataset_path, "tiny-imagenet-200", "train")
data_cfg.dataset_path = train_path
print("Tiny-ImageNet train path:", train_path)

train_ds, val_ds, train_loader, val_loader = create_imagenet_loaders(data_cfg)

print(f"Train samples: {len(train_ds):,}")
print(f"Val   samples: {len(val_ds):,}")
print(f"Classes:       {data_cfg.num_classes}")
print(f"Batches/epoch: train={len(train_loader)}, val={len(val_loader)}")

Tiny-ImageNet train path: /home/rafael/.cache/kagglehub/datasets/akash2sharma/tiny-imagenet/versions/1/tiny-imagenet-200/train
Train samples: 90,000
Val   samples: 10,000
Classes:       200
Batches/epoch: train=1406, val=157


## 3. Model shape check


In [4]:
x = torch.randn(2, 3, data_cfg.img_size, data_cfg.img_size)
for label, ctor in [
    ("AlexNetTV",    build_alexnet),
    ("VGGStyleCNN",  build_vggstylecnn),
    ("ResNet18TV",   build_resnet18),
    ("MobileNetV2TV",build_mobilenetv2),
]:
    m = ctor().eval()
    with torch.no_grad():
        y = m(x)
    assert y.shape == (2, data_cfg.num_classes), f"{label}: unexpected shape {y.shape}"
    info = torchinfo.summary(m, input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)
    print(f"{label:20s} OK -> {tuple(y.shape)}, params={info.total_params/1e6:.2f}M")

AlexNetTV            OK -> (2, 200), params=57.82M
VGGStyleCNN          OK -> (2, 200), params=2.41M
ResNet18TV           OK -> (2, 200), params=11.28M


Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /home/rafael/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth
100%|██████████| 13.6M/13.6M [00:00<00:00, 22.6MB/s]


MobileNetV2TV        OK -> (2, 200), params=2.48M


## 4. Model registration

Fuse maps are index-based paths inside `features` for flat-Sequential models.
`ResNet18TV` and `MobileNetV2TV` are registered but excluded from QAT — their residual
adds use plain `+` / inverted bottleneck without `FloatFunctional`, so fake-quant observers
cannot intercept them.

In [5]:
# Conv-ReLU (torchvision AlexNet has no BN in features)
FUSE_MAP_ALEXNET_TV = [["0","1"],["3","4"],["6","7"],["8","9"],["10","11"]]

# Conv-BN-ReLU, two per VGG stage (5 stages × 2 = 10 groups)
FUSE_MAP_VGG = [
    ["0","1","2"],["3","4","5"],
    ["7","8","9"],["10","11","12"],
    ["14","15","16"],["17","18","19"],
    ["21","22","23"],["24","25","26"],
    ["28","29","30"],["31","32","33"],
]

MODEL_REGISTRY.clear()
register_model("alexnet_tv",  build_alexnet,     fuse_map=FUSE_MAP_ALEXNET_TV, fuse_root_attr="features", lr=3e-4)
register_model("vgg_style",   build_vggstylecnn, fuse_map=FUSE_MAP_VGG,        fuse_root_attr="features", lr=1e-3)
register_model("resnet18_tv", build_resnet18,    fuse_map=[],                                             lr=1e-4)  # ponytail: QAT skipped — vanilla residual adds
register_model("mobilenetv2", build_mobilenetv2, fuse_map=[],                                             lr=1e-4)  # ponytail: QAT skipped — inverted residual adds

print("Registered:", list(MODEL_REGISTRY))

Registered: ['alexnet_tv', 'vgg_style', 'resnet18_tv', 'mobilenetv2']


## 5. FP32 training


In [ ]:
fp32_training_results = {}

for name, spec in MODEL_REGISTRY.items():
    if (SAVE_DIR / f"{name}_best.pth").exists():
        print(f"Skipping FP32 training for {name} (checkpoint found).")
        continue
    model_cfg = replace(fp32_cfg, lr=spec.get("lr", fp32_cfg.lr))
    print("=" * 72)
    print(f"Training: {name}  lr={model_cfg.lr}  epochs={model_cfg.epochs}")
    print("=" * 72)

    model = spec["ctor"]().to(device)

    run = wandb.init(
        project="alexnet-baselines",
        group="fp32-baselines",
        name=f"{name}_fp32",
        config={**asdict(model_cfg), "arch": name, "phase": "fp32"},
        mode="offline",
    )

    trainer = Trainer(
        model, train_loader, val_loader, model_cfg,
        device, SAVE_DIR, name, num_classes=data_cfg.num_classes,
        wandb_run=run,
    )
    results = trainer.fit()
    wandb.finish()

    fp32_training_results[name] = results

    del model
    torch.cuda.empty_cache()

print("\nFP32 training complete.")

## 6. Quantization-Aware Training (QAT)

`ResNet18TV` and `MobileNetV2TV` are excluded: their residual adds (`+` / inverted bottleneck)
aren't interceptable by fake-quant observers without `FloatFunctional`.
QAT trains on GPU; `convert` and INT8 inference run on CPU (fbgemm).

In [ ]:
QAT_SKIP = {"resnet18_tv", "mobilenetv2"}  # ponytail: no FloatFunctional residual adds

qat_train_cfg = replace(
    fp32_cfg,
    epochs=qat_cfg.epochs,
    lr=qat_cfg.lr,
    weight_decay=qat_cfg.weight_decay,
    use_amp=False,  # AMP incompatible with fake-quant observers
)

qat_models = {}
qat_training_results = {}

for name in MODEL_REGISTRY:
    if name in QAT_SKIP:
        print(f"Skipping QAT for {name} (QAT_SKIP).")
        continue

    if (SAVE_DIR / f"qat_{name}_best.pth").exists():
        print(f"Skipping QAT for {name} (checkpoint found).")
        continue

    print("=" * 72)
    print(f"QAT fine-tuning: {name}")
    print("=" * 72)

    qat_model = build_qat(name, save_dir=SAVE_DIR, device=device)
    cb = make_qat_callback(qat_cfg.freeze_bn_epoch, qat_cfg.disable_observer_epoch)

    run = wandb.init(
        project="alexnet-baselines",
        group="qat-baselines",
        name=f"{name}_qat",
        config={
            **asdict(qat_train_cfg),
            "arch": name,
            "phase": "qat",
            "freeze_bn_epoch": qat_cfg.freeze_bn_epoch,
            "disable_observer_epoch": qat_cfg.disable_observer_epoch,
        },
        mode="offline",
    )

    trainer = Trainer(
        qat_model, train_loader, val_loader, qat_train_cfg,
        device, SAVE_DIR, f"qat_{name}", num_classes=data_cfg.num_classes,
        wandb_run=run,
        epoch_callback=cb,
    )
    results = trainer.fit()
    wandb.finish()

    qat_training_results[name] = results
    qat_models[name] = qat_model.cpu()

    del qat_model
    torch.cuda.empty_cache()

print("\nQAT training complete.")

## 7. INT8 conversion & CPU evaluation

`torch.ao.quantization.convert` materialises true quantised ops and must run on CPU.


In [8]:
val_loader_cpu = torch.utils.data.DataLoader(
    val_ds, batch_size=data_cfg.batch_size, shuffle=False, num_workers=0, pin_memory=False,
)
cpu_device = torch.device("cpu")

int8_models = {name: convert_to_int8(m.cpu().eval()) for name, m in qat_models.items()}

for name, m in int8_models.items():
    torch.save(m.state_dict(), SAVE_DIR / f"{name}.pth")
print("INT8 conversion done.")

int8_metrics = {}
for name, m in int8_models.items():
    dummy_cfg = replace(fp32_cfg, use_amp=False)
    trainer = Trainer(
        m.cpu().eval(), val_loader_cpu, val_loader_cpu, dummy_cfg,
        cpu_device, SAVE_DIR, name, num_classes=data_cfg.num_classes,
    )
    metrics = trainer.evaluate(topk=(1, 5))
    int8_metrics[name] = metrics
    print(f"{name:22s} | loss={metrics['loss']:.4f} | top1={metrics['top1']:.2f}% | top5={metrics['top5']:.2f}%")

INT8 conversion done.
alexnet_tv             | loss=3.6944 | top1=18.90% | top5=42.90%
vgg_style              | loss=4.4299 | top1=7.58% | top5=22.98%


## 8. FP32 evaluation — reload best checkpoints


In [9]:
CTORS = {
    "alexnet_tv":  build_alexnet,
    "vgg_style":   build_vggstylecnn,
    "resnet18_tv": build_resnet18,
    "mobilenetv2": build_mobilenetv2,
}

fp32_best = {name: load_best_model(name, ctor, SAVE_DIR, device) for name, ctor in CTORS.items()}

fp32_metrics = {}
for name, m in fp32_best.items():
    dummy_cfg = replace(fp32_cfg, use_amp=False)
    trainer = Trainer(
        m, train_loader, val_loader, dummy_cfg,
        device, SAVE_DIR, name, num_classes=data_cfg.num_classes,
    )
    metrics = trainer.evaluate(topk=(1, 5))
    fp32_metrics[name] = metrics
    print(f"{name:22s} | loss={metrics['loss']:.4f} | top1={metrics['top1']:.2f}% | top5={metrics['top5']:.2f}%")

alexnet_tv             | loss=3.8970 | top1=15.20% | top5=38.46%
vgg_style              | loss=4.4882 | top1=6.49% | top5=22.02%
resnet18_tv            | loss=2.5549 | top1=40.42% | top5=68.30%
mobilenetv2            | loss=2.5513 | top1=41.45% | top5=69.31%


## 9. Final comparison table


In [10]:
rows = []

for name, m in fp32_best.items():
    info = torchinfo.summary(m, input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)
    rows.append({
        "model": name, "precision": "FP32",
        "top1_%": fp32_metrics[name]["top1"],
        "top5_%": fp32_metrics[name]["top5"],
        "loss":   fp32_metrics[name]["loss"],
        "params_M": info.total_params / 1e6,
        "size_MB":  disk_mb(SAVE_DIR / f"{name}_best.pth"),
    })

for name, m in int8_models.items():
    info = torchinfo.summary(fp32_best[name], input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)
    rows.append({
        "model": f"{name}_INT8", "precision": "INT8",
        "top1_%": int8_metrics[name]["top1"],
        "top5_%": int8_metrics[name]["top5"],
        "loss":   int8_metrics[name]["loss"],
        "params_M": info.total_params / 1e6,
        "size_MB":  disk_mb(SAVE_DIR / f"{name}.pth"),
    })

df = build_comparison_table(rows)
df.to_csv(RESULTS_DIR / "final_comparison.csv", index=False)
df

,model,precision,top1_%,top5_%,loss,params_M,size_MB
0,mobilenetv2,FP32,41.452551,69.308913,2.551321,2.480072,28.751802
1,resnet18_tv,FP32,40.420857,68.297613,2.554949,11.279112,129.208559
2,alexnet_tv,FP32,15.198159,38.463736,3.896986,57.823240,661.753958
3,vgg_style,FP32,6.494214,22.020870,4.488163,2.405288,27.584690
4,alexnet_tv_INT8,INT8,18.899468,42.903540,3.694377,57.823240,55.332613
5,vgg_style_INT8,INT8,7.575349,22.977319,4.429891,2.405288,2.376703


In [11]:
print("=" * 72)
print("RANKING BY TOP-1 ACCURACY (all precisions)")
print("=" * 72)
ranked = df.sort_values("top1_%", ascending=False).reset_index(drop=True)
for i, row in ranked.iterrows():
    print(f"{i+1:2d}. {row['model']:26s} [{row['precision']}] | "
          f"top1={row['top1_%']:6.2f}% | top5={row['top5_%']:6.2f}% | "
          f"params={row['params_M']:6.2f}M | size={row['size_MB']:6.2f}MB")

RANKING BY TOP-1 ACCURACY (all precisions)
 1. mobilenetv2                [FP32] | top1= 41.45% | top5= 69.31% | params=  2.48M | size= 28.75MB
 2. resnet18_tv                [FP32] | top1= 40.42% | top5= 68.30% | params= 11.28M | size=129.21MB
 3. alexnet_tv_INT8            [INT8] | top1= 18.90% | top5= 42.90% | params= 57.82M | size= 55.33MB
 4. alexnet_tv                 [FP32] | top1= 15.20% | top5= 38.46% | params= 57.82M | size=661.75MB
 5. vgg_style_INT8             [INT8] | top1=  7.58% | top5= 22.98% | params=  2.41M | size=  2.38MB
 6. vgg_style                  [FP32] | top1=  6.49% | top5= 22.02% | params=  2.41M | size= 27.58MB


## 10. Persist experiment summary


In [12]:
create_results_summary(
    results={
        "fp32_metrics": fp32_metrics,
        "int8_metrics": int8_metrics,
        "fp32_training_results": fp32_training_results,
        "qat_training_results": qat_training_results,
    },
    config=asdict(fp32_cfg),
    output_path=RESULTS_DIR / "experiment_summary.json",
)

## W&B — syncing offline runs

All runs were saved locally with `mode="offline"`. Two run groups are tracked:
- **`fp32-baselines`** — one run per architecture, FP32 training
- **`qat-baselines`** — one run per architecture (excl. `resnet18_tv`), QAT fine-tuning

When ready, sync to the W&B dashboard from a terminal:

```bash
wandb sync --sync-all
```
